# Assignment 3: Document Retrieval Evaluation

**Week 3 - Embeddings, Semantic Search & RAG**

This notebook extracts a sample PDF, creates page-aware chunks, generates vector embeddings, retrieves relevant passages for ten questions, compares two chunk sizes, and evaluates whether the retrieved passages contain the expected answer evidence.

**Backends:** The preferred backend is the dense `sentence-transformers/all-MiniLM-L6-v2` embedding model. If it is unavailable, `auto` mode falls back to a sparse TF-IDF vector representation and prints a warning. Use the same backend for both chunk-size experiments to make the comparison fair.

Run cells from top to bottom.

In [1]:
from pathlib import Path
import sys, json, pandas as pd

# Find the project root even if VS Code starts the notebook in this subfolder.
HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "rag_core.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find rag_core.py. Open the Week 3 project root in VS Code and run this notebook again.")
sys.path.insert(0, str(PROJECT_ROOT))
from rag_core import extract_pdf_pages, chunk_pages, SemanticRetriever

ASSIGNMENT_DIR = PROJECT_ROOT / "Assignment_3_Document_Retrieval_Evaluation"
PDF_PATH = ASSIGNMENT_DIR / "sample_study_notes.pdf"
OUTPUT_PATH = ASSIGNMENT_DIR / "retrieval_evaluation_results.csv"
print("Project root:", PROJECT_ROOT)
print("PDF:", PDF_PATH)

Project root: D:\SkillNexis_LLM_Development_Week_3
PDF: D:\SkillNexis_LLM_Development_Week_3\Assignment_3_Document_Retrieval_Evaluation\sample_study_notes.pdf


## 1. Extract text from the PDF

Page numbers are 1-based PDF page numbers, including the cover page. The PDF is text-based, so OCR is not needed.

In [2]:
pages = extract_pdf_pages(PDF_PATH)
print(f"Extracted {len(pages)} pages with selectable text.")
for p in pages[:3]:
    print(f"\nPDF page {p.page}: {p.text[:280]}...")
assert len(pages) >= 7, "The sample PDF should contain multiple content pages."

Extracted 8 pages with selectable text.

PDF page 1: SkillNexis LLM Development - Week 3 sample notes
PDF page 1
 Introduction to Machine Learning
Sample Study Notes for Week 3: Embeddings, Semantic Search & RAG
This deliberately text-based PDF is supplied as the source document for the retrieval assignment and
mini project. It con...

PDF page 2: SkillNexis LLM Development - Week 3 sample notes
PDF page 2
Introduction to Machine Learning
Learning objectives
These notes introduce machine learning, common learning settings, model evaluation, text embeddings,
and retrieval-augmented generation. The examples are designed for ...

PDF page 3: SkillNexis LLM Development - Week 3 sample notes
PDF page 3
Supervised Learning
Definition
Supervised learning trains a model on labelled examples. Each example contains input features and a
known target, so the model can learn a mapping from inputs to target values.
Classificati...


## 2. Create chunks with page references

The chunker does not merge text across page boundaries. Every chunk keeps the source PDF page number.

In [3]:
SMALL_CHUNK_SIZE = 100
LARGE_CHUNK_SIZE = 180
OVERLAP = 20
small_chunks = chunk_pages(pages, chunk_size=SMALL_CHUNK_SIZE, overlap=OVERLAP)
large_chunks = chunk_pages(pages, chunk_size=LARGE_CHUNK_SIZE, overlap=OVERLAP)
print("Small chunks:", len(small_chunks))
print("Large chunks:", len(large_chunks))
print("Example chunk metadata:", small_chunks[0])

Small chunks: 16
Large chunks: 9
Example chunk metadata: Chunk(chunk_id='p1-c0', text='SkillNexis LLM Development - Week 3 sample notes PDF page 1 Introduction to Machine Learning Sample Study Notes for Week 3: Embeddings, Semantic Search & RAG This deliberately text-based PDF is supplied as the source document for the retrieval assignment and mini project. It contains seven numbered content pages so source references can be checked directly. Topics: machine learning, supervised and unsupervised learning, model evaluation, overfitting, embeddings, similarity search, retrieval-augmented generation, citations, and abstention.', page=1, source='sample_study_notes.pdf', start_word=0, end_word=74)


## 3. Load the ten-question evaluation set

Eight questions have expected evidence in the PDF. Two are intentionally unsupported, which helps test evidence-aware abstention.

In [4]:
questions_df = pd.read_csv(ASSIGNMENT_DIR / "test_questions.csv")
questions_df["gold_pdf_pages"] = questions_df["gold_pdf_pages"].fillna("").apply(lambda x: [int(float(p)) for p in str(x).split(";") if p])
questions_df["answer_terms"] = questions_df["answer_terms"].fillna("").apply(lambda x: [t for t in str(x).split("|") if t])
questions_df["answerable"] = questions_df["answerable"].astype(str).str.lower().eq("true")
questions_df[["question_id", "question", "gold_pdf_pages", "answerable"]]

,question_id,question,gold_pdf_pages,answerable
0,Q01,What is machine learning?,[2],True
1,Q02,How does supervised learning learn from examples?,[3],True
2,Q03,What is the difference between classification ...,[3],True
3,Q04,What does clustering do in unsupervised learning?,[4],True
4,Q05,"What is overfitting, and name two ways to redu...",[5],True
5,Q06,"What are the roles of the training, validation...",[5],True
6,Q07,How is precision calculated and what does it m...,[6],True
7,Q08,"What is a text embedding, and what does cosine...",[7],True
8,Q09,What exact accuracy percentage did the example...,[],False
9,Q10,In what year was the first GPT model released?,[],False


## 4. Generate embeddings and retrieve passages

The retriever embeds all chunks and each question. Cosine similarity ranks candidate passages. The default `auto` mode tries dense sentence embeddings first and falls back to TF-IDF if the model cannot load. If the model download is available, it may take a little longer on the first run.

In [5]:
def run_retrieval(chunks, chunk_size, backend="auto", top_k=3):
    retriever = SemanticRetriever(backend=backend).fit(chunks)
    print(f"Chunk size {chunk_size}: using backend '{retriever.backend}'")
    if retriever.warning:
        print("WARNING:", retriever.warning)
    rows = []
    for _, q in questions_df.iterrows():
        hits = retriever.search(q["question"], top_k=top_k)
        hit_pages = sorted({h["chunk"].page for h in hits})
        combined = " ".join(h["chunk"].text for h in hits).lower()
        # Evidence check: all required answer phrases must occur in retrieved passages.
        # Unsupported questions have no gold answer terms, so their evidence_present is False.
        terms = q["answer_terms"]
        evidence_present = bool(terms) and all(term.lower() in combined for term in terms)
        rows.append({
            "question_id": q["question_id"],
            "question": q["question"],
            "chunk_size_words": chunk_size,
            "backend": retriever.backend,
            "top_1_pdf_page": hits[0]["chunk"].page if hits else None,
            "retrieved_pdf_pages": ";".join(map(str, hit_pages)),
            "top_1_similarity": round(hits[0]["score"], 4) if hits else 0.0,
            "expected_pdf_pages": ";".join(map(str, q["gold_pdf_pages"])),
            "answerable_in_document": bool(q["answerable"]),
            "retrieved_passages_contain_expected_evidence": bool(evidence_present),
            "top_passages": "\n---\n".join(h["chunk"].text for h in hits),
        })
    return rows

results_small = run_retrieval(small_chunks, SMALL_CHUNK_SIZE, backend="auto", top_k=3)
results_large = run_retrieval(large_chunks, LARGE_CHUNK_SIZE, backend="auto", top_k=3)
results = pd.DataFrame(results_small + results_large)
results.head(6)

Chunk size 100: using backend 'tfidf'
Chunk size 180: using backend 'tfidf'


,question_id,question,chunk_size_words,backend,top_1_pdf_page,retrieved_pdf_pages,top_1_similarity,expected_pdf_pages,answerable_in_document,retrieved_passages_contain_expected_evidence,top_passages
0,Q01,What is machine learning?,100,tfidf,2,1;2;7,0.3055,2,True,True,SkillNexis LLM Development - Week 3 sample not...
1,Q02,How does supervised learning learn from examples?,100,tfidf,3,2;3,0.2320,3,True,True,SkillNexis LLM Development - Week 3 sample not...
2,Q03,What is the difference between classification ...,100,tfidf,3,3;6,0.1357,3,True,True,SkillNexis LLM Development - Week 3 sample not...
3,Q04,What does clustering do in unsupervised learning?,100,tfidf,4,1;2;4,0.2131,4,True,True,SkillNexis LLM Development - Week 3 sample not...
4,Q05,"What is overfitting, and name two ways to redu...",100,tfidf,5,3;5,0.1697,5,True,True,SkillNexis LLM Development - Week 3 sample not...
5,Q06,"What are the roles of the training, validation...",100,tfidf,5,5;6,0.2651,5,True,False,SkillNexis LLM Development - Week 3 sample not...


## 5. Compare the two chunk sizes

The evidence-present flag is a simple, reproducible phrase-coverage check against predefined expected answer terms. It is useful for this small assignment but is not a complete semantic entailment judge. Unsupported questions intentionally have no expected terms.

In [6]:
summary = (results.groupby("chunk_size_words")
           .agg(question_count=("question_id", "count"),
                evidence_found_count=("retrieved_passages_contain_expected_evidence", "sum"),
                mean_top1_similarity=("top_1_similarity", "mean"))
           .reset_index())
summary["evidence_found_rate"] = summary["evidence_found_count"] / summary["question_count"]
summary

,chunk_size_words,question_count,evidence_found_count,mean_top1_similarity,evidence_found_rate
0,100,10,7,0.19981,0.7
1,180,10,7,0.18032,0.7


## 6. Inspect the results and save the CSV

Review the retrieved source pages and passage text rather than relying only on a metric. A passage can have a high similarity score without fully supporting an answer.

In [7]:
display_cols = ["question_id", "question", "chunk_size_words", "top_1_pdf_page", "retrieved_pdf_pages", "top_1_similarity", "answerable_in_document", "retrieved_passages_contain_expected_evidence"]
display(results[display_cols])
results.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
summary.to_csv(ASSIGNMENT_DIR / "chunk_size_comparison.csv", index=False, encoding="utf-8")
print("Saved:", OUTPUT_PATH)

,question_id,question,chunk_size_words,top_1_pdf_page,retrieved_pdf_pages,top_1_similarity,answerable_in_document,retrieved_passages_contain_expected_evidence
0,Q01,What is machine learning?,100,2,1;2;7,0.3055,True,True
1,Q02,How does supervised learning learn from examples?,100,3,2;3,0.2320,True,True
2,Q03,What is the difference between classification ...,100,3,3;6,0.1357,True,True
3,Q04,What does clustering do in unsupervised learning?,100,4,1;2;4,0.2131,True,True
4,Q05,"What is overfitting, and name two ways to redu...",100,5,3;5,0.1697,True,True
5,Q06,"What are the roles of the training, validation...",100,5,5;6,0.2651,True,False
6,Q07,How is precision calculated and what does it m...,100,6,6;8,0.1721,True,True
7,Q08,"What is a text embedding, and what does cosine...",100,7,2;7;8,0.2943,True,True
8,Q09,What exact accuracy percentage did the example...,100,6,6;8,0.1267,False,False
9,Q10,In what year was the first GPT model released?,100,7,6;7,0.0839,False,False


Saved: D:\SkillNexis_LLM_Development_Week_3\Assignment_3_Document_Retrieval_Evaluation\retrieval_evaluation_results.csv


## 7. Evaluation notes

Write a short interpretation for your submission:
- Which chunk size returned more of the expected answer phrases?
- Did the smaller chunks give more focused passages? Did the larger chunks retain useful context?
- Which of the two unsupported questions retrieved superficially related but non-answering content?
- Why should a similarity score not be treated as proof that the passage answers the question?

The sample CSVs included in this folder are reference outputs from the lightweight TF-IDF backend. Running the notebook regenerates the result files for the selected embedding backend.